In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# --- IMPORTING FROM OTHER FILES
import sys
import os

# Add path to the package: relativistic_dof
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from parameter_dependence import ParametersAxionMassDependence
from second_interpolation import get_process

## Configure the Setup and Select the Production Channel

In this section, we choose the axion production process to analyze and set flags that control output behavior (e.g., saving plots or data). Additional parameters:

- `file_number`: We define it, to select the process:
  - $0$ : Electron scattering
  - $1$ : Muon decay
  - $2$ : Muon scattering
  - $3$ : Tau decay
  - $4$ : Tau scattering

In [3]:
# --- SELECT FILE, SET FLAGS (MANUAL) ---
file_number = 4
flag_save_plot = True

In [4]:
# --- Define process labels and corresponding filenames for selected production channel ---
title_arr = [
    r"$\bf e$ scattering",
    r"$\bf \mu$ decay", 
    r"$\bf \mu$ scattering",
    r"$\bf \tau$ decay", 
    r"$\bf \tau$ scattering"
]
filename_dist_arr = [
    "Distributions_fa_e_scat.dat", 
    "Distributions_fa_mu_dec.dat", 
    "Distributions_fa_mu_scat.dat",
    "Distributions_fa_tau_dec_extended.dat", #"Distributions_fa_tau_dec.dat" 
    "Distributions_fa_tau_scat_extended.dat", #"Distributions_fa_tau_scat.dat"
]
filename_save_plots_arr = [
    "electron_scattering", 
    "muon_decay", 
    "muon_scattering", 
    "tau_decay", 
    "tau_scattering"
]
####################################################################################################################
filename_dist = f"../../data-new/{filename_dist_arr[file_number]}"
# filename_dist = f"/home/krzysztof/Workspace/Master-coding/real_distribution/data-new/{filename_dist_arr[file_number]}"
filename_save_plots = f"{filename_save_plots_arr[file_number]}"

In [5]:
# --- Initialize CLASS, which deals with mass dependence
parameterDependence = ParametersAxionMassDependence(filename_save_plots, filename_dist)

# --- DISTRIBUTION: f(q)*q^3 ---
distribution = "f(q)_q2"  # "f(q)_q2"
parameterDependence.fit_all_distributions(dist=distribution)

# Set bounds for log(m_a)
m_end, m_start = parameterDependence.get_max_min_ma()
log_m_start = np.log(m_start)
log_m_end = np.log(m_end)
log_m_arr_set = np.linspace(log_m_start, log_m_end, 600)

# --- Interpolation ---
interpolated_A_value = parameterDependence.get_interpolated_function(dist=distribution, parameter="A")(log_m_arr_set)
interpolated_b_value = parameterDependence.get_interpolated_function(dist=distribution, parameter="b")(log_m_arr_set)
interpolated_mu_value = parameterDependence.get_interpolated_function(dist=distribution, parameter="mu")(log_m_arr_set)

# --- Base Data ---
m_arr_input = parameterDependence.get_axion_masses()
log_m_arr_input = np.log(m_arr_input)
fit_parameter = parameterDependence.get_fit_parameters(dist=distribution)

In [6]:
# --- Function Fitting: A(log(m_a)), b(log(m_a)), and μ(log(m_a)) ---
# polynomial
parameterDependence.compute_polynomial_fit(dist=distribution, poly_order=13)

# function fit
# parameterDependence.switch_fitting_function("A", "piecewise")
# parameterDependence.switch_fitting_function("b", "b_function")
# parameterDependence.switch_fitting_function("mu", "exp_decay")
# parameterDependence.compute_functional_fit(dist=distribution)

Polynomial fit coefficients for A: [-1.51441903e-08  3.27158564e-07 -2.12336269e-06 -4.55017587e-07
  4.63558587e-05 -3.91271753e-05 -6.21286011e-04  3.77258264e-04
  6.45629701e-03 -4.76572053e-03 -3.22766044e-02  3.14863603e-02
 -5.47459138e-02  1.56710891e+00]
Polynomial fit coefficients for b: [ 6.70950708e-08 -1.75314080e-06  1.66223999e-05 -5.31562660e-05
 -1.63583116e-04  1.55547693e-03 -2.21661073e-03 -9.57066291e-03
  3.23679837e-02 -1.02035595e-02 -6.67264394e-02  6.52895260e-02
 -3.26316140e-02  1.45368031e-01]
Polynomial fit coefficients for μ: [ 2.69213752e-07 -6.61158105e-06  5.67507964e-05 -1.38565124e-04
 -6.83816351e-04  3.89458321e-03  1.69652968e-03 -4.37879804e-02
  7.27848178e-02  3.43299818e-02 -1.38293761e-01  8.53904736e-02
  1.03590791e-01 -3.75180951e+00]


## Plots: $A(log(m_a))$, $b(log(m_a))$, and $μ(log(m_a))$

In [7]:
# --- PLOT EACH PARAMETER SEPARATELY ---
name_of_parameters = ["A", "b", "mu"]
interpolated_param = [interpolated_A_value, interpolated_b_value, interpolated_mu_value]
points_param = [fit_parameter['A'], fit_parameter['b'], fit_parameter['mu']]
titles = [r"$A(\log(m_{a}))$", r"$b(\log(m_{a}))$", r"$-\mu(\log(m_{a}))$"]

colors = ['blue', 'green', 'purple']  # Different colors for better distinction

for i in range(3):
    plt.figure(figsize=(10, 8))
    plt.title(f"{title_arr[file_number]}, {titles[i]}", fontsize=16)

    # Scatter plot of extracted parameter values
    if name_of_parameters[i] == "mu":
        plt.scatter(log_m_arr_input, (-1)*points_param[i],
                    color=colors[i], marker='o', edgecolors='black',
                    alpha=0.8, label='Extracted Parameters')
    else:
        plt.scatter(log_m_arr_input, points_param[i],
                    color=colors[i], marker='o', edgecolors='black',
                    alpha=0.8, label='Extracted Parameters')


    # Plot interpolation of extracted values
    if name_of_parameters[i] == "mu":
        plt.plot(log_m_arr_set, (-1)*interpolated_param[i],
                 linestyle='--', linewidth=2, color='black',
                 label='Interpolated Data')
    else:
        plt.plot(log_m_arr_set, interpolated_param[i],
                 linestyle='--', linewidth=2, color='black',
                 label='Interpolated Data')


    # Plot fitted function for the parameter
    if name_of_parameters[i] == "mu":
        plt.plot(log_m_arr_set, (-1)*parameterDependence.get_fitted_parameter_value(name_of_parameters[i],
                                                                                    log_m_arr_set,
                                                                                  "test_polynomial"),  # test_polynomial, test_fit
                 linestyle='-.', linewidth=2, color='red',
                 label='Fit Function')
    else:
        plt.plot(log_m_arr_set, parameterDependence.get_fitted_parameter_value(name_of_parameters[i],
                                                                               log_m_arr_set,
                                                                             "test_polynomial"),  # test_polynomial, test_fit
                 linestyle='-.', linewidth=2, color='red',
                 label='Fit Function')

    # Labels and formatting
    plt.xlabel(r'$\log(m_{a})$', fontsize=12)
    plt.ylabel(titles[i], fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10)

    # Set log scale only for the y-axis of μ(m_a)
    # if name_of_parameters[i] == "mu":
    #     plt.yscale("log")

    # Show each plot separately
    if flag_save_plot:
        plt.savefig(f'{filename_save_plots}_fit_parameter_{name_of_parameters[i]}.png', format='png', dpi=300)
        plt.close()
    else:
        plt.show()

In [8]:
for i in range(0, len(m_arr_input)):
    ma = m_arr_input[i]
    val_parameter_A = parameterDependence.get_fitted_parameter_value("A", np.log(ma), "test_polynomial")  # test_polynomial, test_fit
    val_parameter_b = parameterDependence.get_fitted_parameter_value("b", np.log(ma), "test_polynomial")
    val_parameter_mu = parameterDependence.get_fitted_parameter_value("mu", np.log(ma), "test_polynomial")

    print("--------------------------------------------------------------------------------------------------------")
    print(f"input A: {fit_parameter['A'][i]}, fitted A: {val_parameter_A}")
    print(f"input b: {fit_parameter['b'][i]}, fitted b: {val_parameter_b}")
    print(f"input mu: {fit_parameter['mu'][i]}, fitted mu: {val_parameter_mu}")


--------------------------------------------------------------------------------------------------------
input A: 1.0329831251039217, fitted A: 1.0316615267245004
input b: -0.09508302650494148, fitted b: -0.08904513648276352
input mu: -2.6346379378220868, fitted mu: -2.605196782732756
--------------------------------------------------------------------------------------------------------
input A: 1.0338519255462282, fitted A: 1.0333373997749074
input b: -0.0964889141734408, fitted b: -0.09349048637930105
input mu: -2.6413962678811576, fitted mu: -2.627088139749043
--------------------------------------------------------------------------------------------------------
input A: 1.034728970726856, fitted A: 1.0347981105004536
input b: -0.09789604192906463, fitted b: -0.09717066615398043
input mu: -2.648216755118201, fitted mu: -2.645215454251122
--------------------------------------------------------------------------------------------------------
input A: 1.0356249326652518, fitted A: 1